# Corpus overview

Exploratory **data-quality** and **descriptive** notebook for a local
BoardGameGeek corpus produced by `board-game-ingest`.

This is **not** a predictive analysis. It does not infer fun, quality,
popularity, or what causes ratings or complexity. Plots do **not** support
inference about all board games or causal conclusions.

`bgg_boardgames_v1` is a curated **convenience / coverage sample**.
It is not a probability sample, not representative of BGG, and not a
rank snapshot. BGG mechanic and category labels are **source facts**.

The notebook prefers a v1 JSONL if present, otherwise v0.


In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from board_game_analysis.ingestion.quality import (
    category_counts,
    flag_anomalies,
    integrity_report,
    list_length_summaries,
    load_games_jsonl,
    load_manifest,
    mechanic_counts,
    missingness_rows,
    numeric_summaries,
)


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "board_game_analysis"
        ).is_dir():
            return candidate
    raise FileNotFoundError("could not locate repository root")


def table(headers: list[str], rows: list[list[object]]) -> None:
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    for row in rows:
        lines.append("| " + " | ".join(str(cell) for cell in row) + " |")
    display(Markdown("\n".join(lines)))


ROOT = repo_root()
V1_JSONL = ROOT / "data" / "processed" / "boardgamegeek" / "corpus_v1.jsonl"
V1_MANIFEST = ROOT / "data" / "derived" / "corpus" / "bgg_boardgames_v1.manifest.json"
V0_JSONL = ROOT / "data" / "processed" / "boardgamegeek" / "corpus_v0.jsonl"
V0_MANIFEST = ROOT / "data" / "derived" / "corpus" / "bgg_boardgames_v0.manifest.json"
RAW_DIR = ROOT / "data" / "raw" / "boardgamegeek"

if V1_JSONL.is_file():
    CORPUS_ID = "bgg_boardgames_v1"
    JSONL = V1_JSONL
    MANIFEST = V1_MANIFEST
else:
    CORPUS_ID = "bgg_boardgames_v0"
    JSONL = V0_JSONL
    MANIFEST = V0_MANIFEST

print(f"repo root: {ROOT}")
print(f"corpus:    {CORPUS_ID}")
print(f"jsonl:     {JSONL}")
print(f"manifest:  {MANIFEST}")
print(f"exists:    jsonl={JSONL.is_file()} manifest={MANIFEST.is_file()}")


## 1. Load and validate

Records are loaded from the processed JSONL and validated with the canonical
Pydantic `Game` model. Invalid lines are **kept and listed**, not dropped.


In [ ]:
if not JSONL.is_file():
    raise FileNotFoundError(
        f"missing {JSONL}. Run: uv run board-game-ingest --corpus "
        "--corpus-id bgg_boardgames_v1"
    )

loaded = load_games_jsonl(JSONL)
manifest = load_manifest(MANIFEST) if MANIFEST.is_file() else None
games = loaded.games
n = len(games)
if manifest and manifest.get("finished_at"):
    CURRENT_YEAR = datetime.fromisoformat(manifest["finished_at"]).year
else:
    CURRENT_YEAR = 2026

display(Markdown(f"**Corpus:** `{CORPUS_ID}`"))
display(Markdown(f"**Validated `Game` records (n):** {n}"))
if loaded.errors:
    display(Markdown("**Load / validation errors (not discarded):**"))
    table(
        ["line", "id", "reason"],
        [
            [err.line_number, err.raw_id or "", err.reason.replace("\n", " ")]
            for err in loaded.errors
        ],
    )
else:
    display(Markdown("Every JSONL line validated as `Game`."))


## 2. Corpus integrity

These checks describe **this file**, not BGG as a whole.


In [ ]:
report = integrity_report(loaded, manifest, current_year=CURRENT_YEAR)
integrity_rows = [
    ["JSONL records (objects)", report.n_jsonl_records],
    ["Validated Game records", n],
    ["Unique game IDs", report.n_unique_ids],
    ["Unique titles", report.n_unique_titles],
    ["Duplicate IDs", ", ".join(report.duplicate_ids) or "none"],
    ["Duplicate titles", ", ".join(report.duplicate_titles) or "none"],
    ["Manifest requested", report.n_manifest_requested],
    ["Manifest ok", report.n_manifest_ok],
    ["Manifest skipped", report.n_manifest_skipped],
    ["Manifest errors", report.n_manifest_errors],
    [
        "JSONL ids missing from manifest",
        ", ".join(report.jsonl_ids_missing_from_manifest) or "none",
    ],
    [
        "Manifest ok ids missing from JSONL",
        ", ".join(report.manifest_ok_ids_missing_from_jsonl) or "none",
    ],
    ["Manifest item types", report.manifest_item_types or "n/a"],
    ["Provenance issues", len(report.provenance_issues)],
    [
        "Title heuristic: expansion-like",
        ", ".join(report.expansion_like_titles) or "none",
    ],
    ["Anomaly codes", report.anomaly_counts or "none"],
    ["Empty taxonomy", ", ".join(report.empty_taxonomy) or "none"],
    ["Duplicate mechanic ids", ", ".join(report.duplicate_mechanic_ids) or "none"],
    ["Non-boardgame ok rows", ", ".join(report.non_boardgame_ok_rows) or "none"],
]
table(["Check", "Value"], integrity_rows)

if report.provenance_issues:
    display(Markdown("**Provenance issues:**"))
    for issue in report.provenance_issues:
        display(Markdown(f"- {issue}"))


## 3. Missingness


In [ ]:
rows = missingness_rows(games)
table(
    ["field", "kind", "present / non-empty", "null", "% null", "empty list", "% empty"],
    [
        [
            row.field,
            row.kind,
            row.present,
            row.missing_null,
            f"{row.pct_missing_null:.1f}",
            row.empty_list if row.empty_list is not None else "—",
            f"{row.pct_empty_list:.1f}" if row.pct_empty_list is not None else "—",
        ]
        for row in rows
    ],
)


## 4. Descriptive distributions

These describe the curated sample only.


In [ ]:
summaries = numeric_summaries(games)
table(
    ["field", "n", "min", "median", "mean", "max", "stdev"],
    [
        [
            row.field,
            row.count,
            "—" if row.minimum is None else round(row.minimum, 4),
            "—" if row.median is None else round(row.median, 4),
            "—" if row.mean is None else round(row.mean, 4),
            "—" if row.maximum is None else round(row.maximum, 4),
            "—" if row.stdev is None else round(row.stdev, 4),
        ]
        for row in summaries
    ],
)


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 5.5))
fig.suptitle(f"Numeric distributions ({CORPUS_ID}, n={n})", fontsize=12)


def hist(ax, values, title, *, bins=10, log_x=False):
    ax.hist(values, bins=bins, color="#4C6A92", edgecolor="white")
    ax.set_title(title, fontsize=10)
    ax.set_ylabel("games")
    if log_x:
        ax.set_xscale("log")
        ax.set_xlabel("log scale")


hist(axes[0, 0], [g.release_year for g in games if g.release_year], "release_year")
hist(
    axes[0, 1],
    [g.min_players for g in games if g.min_players is not None],
    "min_players",
)
hist(
    axes[0, 2],
    [g.max_players for g in games if g.max_players is not None],
    "max_players",
)
hist(
    axes[0, 3],
    [g.min_play_time_minutes for g in games if g.min_play_time_minutes is not None],
    "min play time",
)
hist(
    axes[1, 0],
    [g.max_play_time_minutes for g in games if g.max_play_time_minutes is not None],
    "max play time",
)
hist(axes[1, 1], [g.rating for g in games if g.rating is not None], "rating (1–10)")
hist(
    axes[1, 2],
    [g.rating_count for g in games if g.rating_count is not None],
    "rating_count (log x)",
    log_x=True,
)
hist(
    axes[1, 3],
    [g.complexity for g in games if g.complexity is not None],
    "complexity (1–5)",
)
fig.tight_layout()
plt.show()


## 5. Mechanics and categories

Source-label frequencies. Not a project taxonomy.


In [ ]:
length_rows = list_length_summaries(games)
table(
    ["field", "n", "min", "median", "mean", "max"],
    [
        [
            row.field,
            row.count,
            row.minimum,
            round(row.median, 2),
            round(row.mean, 2),
            row.maximum,
        ]
        for row in length_rows
    ],
)

cats = category_counts(games)
mechs = mechanic_counts(games)
display(Markdown(f"**Distinct BGG categories:** {len(cats)}"))
table(["category (BGG label)", "games"], [list(item) for item in cats[:15]])
display(Markdown(f"**Distinct BGG mechanics:** {len(mechs)}"))
table(["mechanic (BGG label)", "games"], [list(item) for item in mechs[:15]])


## 6. Pairwise views

Descriptive scatter only. No fitted models and no causal claims.


In [ ]:
complexity_rating = [
    (game.complexity, game.rating)
    for game in games
    if game.complexity is not None and game.rating is not None
]
year_complexity = [
    (game.release_year, game.complexity)
    for game in games
    if game.release_year is not None and game.complexity is not None
]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle(f"Pairwise descriptive views ({CORPUS_ID}, n={n})", fontsize=12)
if complexity_rating:
    xs, ys = zip(*complexity_rating, strict=True)
    axes[0].scatter(xs, ys, alpha=0.6, color="#4C6A92")
axes[0].set_xlabel("complexity")
axes[0].set_ylabel("rating")
axes[0].set_title("complexity vs rating")
if year_complexity:
    xs, ys = zip(*year_complexity, strict=True)
    axes[1].scatter(xs, ys, alpha=0.6, color="#4C6A92")
axes[1].set_xlabel("release_year")
axes[1].set_ylabel("complexity")
axes[1].set_title("year vs complexity")
fig.tight_layout()
plt.show()


## 7. Heuristic anomalies

Flags are for review, not rejection. Large publisher lists are common on BGG.


In [ ]:
flags = flag_anomalies(games, current_year=CURRENT_YEAR)
display(Markdown(f"**Flags raised:** {len(flags)} (n={n})"))
if flags:
    table(
        ["id", "title", "code", "detail"],
        [[item.game_id, item.title, item.code, item.detail] for item in flags[:40]],
    )
else:
    display(Markdown("No heuristic flags."))


Findings here describe only the games in the loaded JSONL. They do not
generalize to all board games and they do not explain ratings.
